# Silver — ecommerce_rastreamento_entregas

Este notebook lê a camada Bronze de rastreamento, aplica as 10 regras de qualidade/negócio, salva a tabela Silver em Delta e registra o resumo das falhas em `squad1.dq_monitoring_logs`.

In [0]:

%run ../utils/utils

In [0]:
import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from functools import reduce
from datetime import datetime, timezone

# Variáveis do Processo
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_rastreamento"
TABELA_DQ = "dq_monitoring_logs"

print(f"Iniciando processamento Silver - Rastreamento - Run ID: {RUN_ID}")

##  Anti-Join e Tabelas de Referência


In [0]:
# 1. Carrega a tabela Bronze de Rastreamento
try:
    df_bronze_rastreamento = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: A tabela Bronze de {TABELA_ALVO} não foi encontrada. Rode a Bronze primeiro!")

# 2. Isola o Micro-lote (Considerando Silver E Quarentena)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    
    # Busca na subpasta oficial de quarentena
    if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
        df_quarentena = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
        df_processados = df_silver_atual.select("id_rastreamento") \
            .union(df_quarentena.select("id_rastreamento"))
    else:
        df_processados = df_silver_atual.select("id_rastreamento")
        
    df_micro_lote = df_bronze_rastreamento.join(df_processados, "id_rastreamento", "left_anti")
else:
    df_micro_lote = df_bronze_rastreamento

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote para processar: {qtd_novos}")

# =================================================================================
# 3. Leitura da Tabela de Referência Cruzada (PEDIDOS DA SILVER)
# =================================================================================
if delta_existe("silver", "ecommerce_pedidos", STORAGE_OPTIONS):
    # Forçamos a seleção e o dropDuplicates usando chaves explícitas
    df_pedidos_ref = ler_delta("silver", "ecommerce_pedidos", STORAGE_OPTIONS) \
        .select(
            F.col("id_pedido").cast("long").alias("id_pedido_ref"),
            F.col("status_pedido").cast("string").alias("status_pedido_ref")
        ).dropDuplicates(["id_pedido_ref"])
else:
    schema_pedidos = StructType([
        StructField("id_pedido_ref", LongType(), True),
        StructField("status_pedido_ref", StringType(), True)
    ])
    df_pedidos_ref = spark.createDataFrame([], schema_pedidos)

print("Tabela de referência df_pedidos_ref carregada com sucesso da camada Silver.")

## Aplicação das 10 Regras de Qualidade

In [0]:
if qtd_novos > 0:
    from functools import reduce
    
    # Domínio de status logísticos permitidos (Regra 2)
    status_logísticos_validos = ["em separacao", "coletado", "em transito", "saiu para entrega", "entregue"]
    
    # 1. ETAPA DE ACOPLAMENTO: Adicionado F.lower() e F.trim() para limpar o status da Bronze
    df_acoplado = df_micro_lote \
        .withColumn("dt_evento_ts", F.col("dt_evento").cast("timestamp")) \
        .withColumn("status_entrega", F.lower(F.trim(F.col("status_entrega")))) \
        .join(df_pedidos_ref, F.col("id_pedido_ecommerce") == F.col("id_pedido_ref"), "left_outer")

    # Janelas analíticas declaradas explicitamente sobre o dataframe estável
    w_id_rastreio = Window.partitionBy("id_rastreamento")
    w_cronologia_pedido = Window.partitionBy("id_pedido_ecommerce").orderBy("dt_evento_ts")
    w_transportadora_pedido = Window.partitionBy("id_pedido_ecommerce")

    # 2. ETAPA ANALÍTICA: Computa as transformações de janela
    df_base = df_acoplado \
        .withColumn("qtd_id_rastreio", F.count("*").over(w_id_rastreio)) \
        .withColumn("dt_evento_anterior", F.lag("dt_evento_ts").over(w_cronologia_pedido)) \
        .withColumn("qtd_transportadoras_pedido", F.size(F.collect_set("id_transportadora").over(w_transportadora_pedido)))

    # --- APLICAÇÃO MASSIFICA DAS SUAS 10 REGRAS OFICIAIS ---
    df_silver_rastreio = df_base \
        .withColumn("r1_id_rastreamento_falhou", F.col("id_rastreamento").isNull() | (F.col("id_rastreamento").cast("string") == "") | (F.col("qtd_id_rastreio") > 1)) \
        .withColumn("r2_status_entrega_falhou", F.col("status_entrega").isNull() | (~F.col("status_entrega").isin(status_logísticos_validos))) \
        .withColumn("r3_id_pedido_fk_falhou", F.col("id_pedido_ecommerce").isNull() | F.col("id_pedido_ref").isNull()) \
        .withColumn("r4_dt_evento_falhou", F.col("dt_evento_ts").isNull() | (F.col("dt_evento_ts") > F.current_timestamp())) \
        .withColumn("r5_codigo_rastreio_falhou", F.col("codigo_rastreio").isNull() | (~F.col("codigo_rastreio").rlike(r"^[A-Z]{2}\d{9}$"))) \
        .withColumn("r6_ordem_cronologica_falhou", F.col("dt_evento_anterior").isNotNull() & (F.col("dt_evento_ts") < F.col("dt_evento_anterior"))) \
        .withColumn("r7_consistencia_entrega_falhou", (F.col("status_pedido_ref") == "Entregue") & (F.col("status_entrega") != "entregue")) \
        .withColumn("r8_sla_violado_falhou", (F.col("status_entrega") == "entregue") & (F.datediff(F.col("dt_evento_ts"), F.coalesce(F.col("dt_evento_anterior"), F.col("dt_evento_ts"))) > 30)) \
        .withColumn("r9_pedido_cancelado_movimentado_falhou", (F.col("status_pedido_ref") == "Cancelado") & F.col("id_rastreamento").isNotNull()) \
        .withColumn("r10_transportadora_inconsistente_falhou", (F.col("status_pedido_ref") != "Cancelado") & (F.col("qtd_transportadoras_pedido") > 1))

    # Separação por Severidade baseada nas colunas acima
    regras_criticas = [
        "r1_id_rastreamento_falhou", "r2_status_entrega_falhou", "r3_id_pedido_fk_falhou", 
        "r4_dt_evento_falhou", "r6_ordem_cronologica_falhou", "r9_pedido_cancelado_movimentado_falhou"
    ]
    regras_avisos = ["r5_codigo_rastreio_falhou", "r7_consistencia_entrega_falhou", "r8_sla_violado_falhou", "r10_transportadora_inconsistente_falhou"]

    condicao_falha_critica = reduce(lambda a, b: a | b, [F.col(c) for c in regras_criticas])
    condicao_aviso = reduce(lambda a, b: a | b, [F.col(c) for c in regras_avisos]) if regras_avisos else F.lit(False)

    df_silver_rastreio = df_silver_rastreio \
        .withColumn("silver_linha_valida", ~condicao_falha_critica) \
        .withColumn("silver_tem_aviso", condicao_aviso) \
        .withColumn("silver_processed_at", F.current_timestamp()) \
        .withColumn("silver_run_id", F.lit(RUN_ID))
    
    print("Muralha de qualidade de rastreamento aplicada com sucesso com a quebra corrigida!")
else:
    print("Nenhum dado logístico novo para aplicar regras.")

## Catálogo de Regras e Logs

In [0]:
if qtd_novos > 0:
    catalogo_regras = [
        {"coluna": "r1_id_rastreamento_falhou", "regra": "R1_ID_RASTREIO_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_status_entrega_falhou", "regra": "R2_STATUS_ENTREGA_INVALIDO", "severidade": "Critica"},
        {"coluna": "r3_id_pedido_fk_falhou", "regra": "R3_ID_PEDIDO_FK_INEXISTENTE", "severidade": "Critica"},
        {"coluna": "r4_dt_evento_falhou", "regra": "R4_DATA_EVENTO_FUTURA_NULA", "severidade": "Critica"},
        {"coluna": "r5_codigo_rastreio_falhou", "regra": "R5_FORMATO_COD_RASTREIO_INVALIDO", "severidade": "Aviso"},
        {"coluna": "r6_ordem_cronologica_falhou", "regra": "R6_INVERSAO_CRONOLOGICA_STATUS", "severidade": "Critica"},
        {"coluna": "r7_consistencia_entrega_falhou", "regra": "R7_DIVERGENCIA_STATUS_ENTREGUE", "severidade": "Aviso"},
        {"coluna": "r8_sla_violado_falhou", "regra": "R8_SLA_LOGISTICO_MAIOR_30_DIAS", "severidade": "Aviso"},
        {"coluna": "r9_pedido_cancelado_movimentado_falhou", "regra": "R9_PEDIDO_CANCELADO_COM_EVENTO", "severidade": "Critica"},
        {"coluna": "r10_transportadora_inconsistente_falhou", "regra": "R10_MULTIPLAS_TRANSPORTADORAS_PEDIDO", "severidade": "Aviso"}
    ]

    total_registros = df_silver_rastreio.count()
    logs_list = []

    for r in catalogo_regras:
        qtd_falhas = df_silver_rastreio.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID, TABELA_ALVO, r["regra"], "FAIL", r["severidade"],
                int(qtd_falhas), int(total_registros), datetime.now(timezone.utc), f"Bronze Delta ({TABELA_ALVO})"
            ))

    # Schema dos logs (com fallback de segurança)
    schema_final = StructType([
        StructField("run_id", StringType(), True),
        StructField("tabela", StringType(), True),
        StructField("regra", StringType(), True),
        StructField("status", StringType(), True),
        StructField("severidade", StringType(), True),
        StructField("qtd_registros_falhos", IntegerType(), True),
        StructField("qtd_registros_total", IntegerType(), True),
        StructField("timestamp_execucao", TimestampType(), True),
        StructField("arquivo_origem", StringType(), True)
    ])

    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema=schema_final)
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema=schema_final)

    # --- AJUSTE: Unificação das condições para a Quarentena ---
    flags_criticas = [r["coluna"] for r in catalogo_regras if r["severidade"] == "Critica"]
    flags_avisos = [r["coluna"] for r in catalogo_regras if r["severidade"] == "Aviso"]

    condicao_falha_critica = reduce(lambda a, b: a | b, [F.col(c) for c in flags_criticas])
    condicao_aviso = reduce(lambda a, b: a | b, [F.col(c) for c in flags_avisos]) if flags_avisos else F.lit(False)
    
    # Qualquer falha (Crítica ou Aviso) invalida a linha para a Silver
    condicao_total_falha = condicao_falha_critica | condicao_aviso

    df_silver_rastreio = df_silver_rastreio \
        .withColumn("silver_linha_valida", ~condicao_total_falha) \
        .withColumn("silver_tem_aviso", condicao_aviso) \
        .withColumn("silver_processed_at", F.current_timestamp()) \
        .withColumn("silver_run_id", F.lit(RUN_ID))

    print("Logs calculados e Muralha de Qualidade ajustada: Avisos agora invalidam a linha para a Silver.")
else:
    # Schema vazio de fallback
    schema_final_vazio = StructType([
        StructField("run_id", StringType(), True),
        StructField("tabela", StringType(), True),
        StructField("regra", StringType(), True),
        StructField("status", StringType(), True),
        StructField("severidade", StringType(), True),
        StructField("qtd_registros_falhos", IntegerType(), True),
        StructField("qtd_registros_total", IntegerType(), True),
        StructField("timestamp_execucao", TimestampType(), True),
        StructField("arquivo_origem", StringType(), True)
    ])
    
    df_dq_monitoring_logs_novos = spark.createDataFrame([], schema=schema_final_vazio)
    print("Micro-lote vazio: Nenhum log de monitoramento gerado.")

## Gravação Final via SDK

In [0]:
if qtd_novos > 0:
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id", "silver_tem_aviso"]
    
    # ---------------- 1. GRAVAÇÃO DOS VÁLIDOS ---------------- #
    # Certifique-se de que df_silver_rastreio é a variável correta carregada acima
    df_silver_validos = df_silver_rastreio \
        .filter(F.col("silver_linha_valida") == True) \
        .select(*colunas_finais)
        
    qtd_validos = df_silver_validos.count()
    print(f"Registros aprovados para a Silver: {qtd_validos}")

    if qtd_validos > 0:
        sucesso_silver = gravar_delta(
            df=df_silver_validos, camada="silver", tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS, mode="append", particionar=True
        )
        if sucesso_silver:
            print(f"Tabela Silver {TABELA_ALVO} atualizada com sucesso!")

    # ---------------- 2. GRAVAÇÃO DA QUARENTENA (DEDUPLICADA) ---------------- #
    df_silver_invalidos = df_silver_rastreio \
        .filter(F.col("silver_linha_valida") == False) \
        .select(*colunas_finais)
        
    # Lógica única de gravação de quarentena
    if df_silver_invalidos.count() > 0:
        # A. Verifica histórico para deduplicar
        if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
            df_quarentena_historico = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
            # Dedup pelo ID da sua tabela (ex: id_cliente ou id_rastreamento)
            df_quarentena_para_gravar = df_silver_invalidos.join(
                df_quarentena_historico.select("id_rastreamento"), 
                on="id_rastreamento", 
                how="left_anti"
            )
        else:
            df_quarentena_para_gravar = df_silver_invalidos

        # B. Grava apenas o que sobrou (o que não existe no histórico)
        qtd_novos_rejeitados = df_quarentena_para_gravar.count()
        if qtd_novos_rejeitados > 0:
            sucesso_quarentena = gravar_delta(
                df=df_quarentena_para_gravar,
                camada="silver/quarentena",
                tabela=TABELA_ALVO,
                storage_opts=STORAGE_OPTIONS,
                mode="append",
                particionar=False 
            )
            if sucesso_quarentena:
                print(f"Enviados {qtd_novos_rejeitados} registros novos para a quarentena.")
        else:
            print("Todos os registros reprovados já existiam na quarentena histórica.")

    # ---------------- 3. GRAVAÇÃO DOS LOGS NA RAIZ ---------------- #
    # Garante que logs vazios não quebrem o processamento
    if 'df_dq_monitoring_logs_novos' in locals() and df_dq_monitoring_logs_novos.count() > 0:
        sucesso_logs = gravar_delta(
            df=df_dq_monitoring_logs_novos, 
            camada="", 
            tabela=TABELA_DQ,
            storage_opts=STORAGE_OPTIONS, 
            mode="append", 
            particionar=False
        )
        if sucesso_logs:
            print("Logs de qualidade consolidados na raiz!")
else:
    print("Rotina finalizada sem alterações físicas.")

##  VALIDACAO


In [0]:
print("===== VALIDAÇÃO FINAL =====")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.orderBy(F.col("silver_processed_at").desc()).limit(20))
else:
    print(f"A tabela Silver {TABELA_ALVO} ainda não existe.")

if delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
    df_logs_validacao = ler_delta(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS)
    df_logs_filtrados = df_logs_validacao.filter(F.col("tabela") == TABELA_ALVO)
    
    print(f"Logs na {TABELA_DQ} para {TABELA_ALVO}:", df_logs_filtrados.count())
    display(df_logs_filtrados.orderBy(F.col("timestamp_execucao").desc()).limit(20))
else:
    print(f"Tabela {TABELA_DQ} ainda não existe no Data Lake.")